# 🚀 Ejemplo de Flujo de Trabajo Completo

Este notebook muestra cómo usar los módulos de la plantilla para un flujo completo:
1. Cargar y procesar datos
2. Entrenar modelo
3. Evaluar modelo
4. Guardar resultados

In [ ]:
import sys
sys.path.append('../..')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

## 1. Procesamiento de Datos

Usa el pipeline de datos para cargar y procesar.

In [ ]:
from scripts.data_processing import process_data

# Configuración del pipeline
config = {
    'date_columns': ['fecha_registro'],
    'categorical_columns': ['genero', 'ciudad'],
    'numerical_columns': ['edad', 'ingreso']
}

# Procesar datos
df = process_data(
    input_path='../../data/raw/clientes.csv',
    output_path='../../data/processed/features.parquet',
    config=config,
    validate=True
)

print(f"Datos procesados: {df.shape}")
df.head()

## 2. Preparar Train/Test Split

In [ ]:
# Separar features y target
X = df.drop('target', axis=1)
y = df['target']

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 3. Entrenar Modelo

Usa la función de entrenamiento para entrenar y guardar.

In [ ]:
from scripts.model_training import train_model

# Crear modelo
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

# Entrenar y guardar
trained_model = train_model(
    X_train, y_train,
    model=model,
    model_name='churn_rf_baseline',
    output_dir='../../models/experiments',
    save=True
)

## 4. Evaluar Modelo

Calcula métricas de evaluación.

In [ ]:
from scripts.model_evaluation import evaluate_classification_model

# Predicciones
y_pred = trained_model.predict(X_test)
y_pred_proba = trained_model.predict_proba(X_test)[:, 1]

# Evaluar
metrics = evaluate_classification_model(
    y_true=y_test,
    y_pred=y_pred,
    y_pred_proba=y_pred_proba
)

print("\n📊 Métricas de Evaluación:")
for metric, value in metrics.items():
    if metric != 'confusion_matrix':
        print(f"  {metric}: {value:.3f}")

## 5. Guardar Metadata del Modelo

In [ ]:
from scripts.model_training import save_model_metadata

metadata = {
    'model_name': 'churn_rf_baseline',
    'version': '1.0.0',
    'algorithm': 'RandomForest',
    'hyperparameters': {
        'n_estimators': 100,
        'max_depth': 10,
        'random_state': 42
    },
    'metrics': metrics,
    'features': list(X_train.columns),
    'training_samples': len(X_train)
}

save_model_metadata(
    model_path='../../models/experiments/churn_rf_baseline.pkl',
    metadata=metadata
)

print("✅ Metadata guardada")

## 6. (Opcional) Notificar Resultados

In [ ]:
from scripts.model_evaluation import notify_evaluation_results

# Enviar notificación a Slack (si está configurado)
notify_evaluation_results(
    metrics=metrics,
    model_name='Churn RF Baseline v1.0'
)

## 📝 Próximos Pasos

1. Documentar experimento en `docs/model/experimentation_log.md`
2. Si el modelo es bueno, actualizar `docs/model/model_card.md`
3. Mover modelo a `models/production/` si va a producción
4. Seguir guía de deployment en `docs/ops/deployment.md`